# Vehicle Fuel Efficiency Prediction - Final Notebook
Integration notebook. Reusable code lives in `src/vehicle_efficiency/`; this notebook calls it and explains the results.

| Section | Owner | Status |
|---|---|---|
| 1. Problem | Aktore | draft below - review |
| 2. Dataset | Aktore | draft below - review |
| 3. EDA | Aktore | TODO (copy final plots from `01_eda.ipynb`) |
| 4. Cleaning | Khamza | written from real output - awaiting teammate review |
| 5. Feature engineering | Khamza | validated on validation set + CV - awaiting teammate review |
| 6. Splitting | Khamza | executable |
| 7. Baseline | Khamza | written - awaiting teammate review |
| 8. Models | Khamza (LR: written), Askhat (Tree, KNN, CV: written) | awaiting teammate review |
| 9. Evaluation | Askhat | validation comparison, error analysis and single test run written - awaiting review |
| 10. Conclusions | all | TODO |
| 11. Final-stage plan | Aktore | TODO |

Only mark a section done when it has been reviewed by a teammate.

In [1]:
import pandas as pd
from vehicle_efficiency.data import download_raw, load_dataset, load_raw, make_splits, prepare
from vehicle_efficiency.evaluation import cross_validate_pipeline, evaluate
from vehicle_efficiency.models import (build_baseline, build_decision_tree,
                                       build_knn, build_linear_regression)
from vehicle_efficiency.preprocessing import add_features, missing_summary, split_xy

results = {}  # model name -> validation metrics, filled in below

## 1. Problem
Predict a vehicle's fuel efficiency (**MPG**, EPA combined miles per gallon; higher = more efficient) from attributes such as cylinders, engine displacement, model year, drive and transmission. Weight and horsepower are not available in this dataset (see `data/README.md`). This is a **regression** problem.

**Prediction scenario:** estimating the combined MPG of a recent (model year 2015+) gasoline, diesel, hybrid or flex-fuel vehicle from its specification. TODO (Aktore): refine the problem statement in your own words.

**Metrics:** MAE (primary, in MPG: average absolute error), RMSE (in MPG: penalises large errors more), R² (unitless: share of variance explained relative to predicting the mean).

## 2. Dataset
US EPA / DOE *fueleconomy.gov* vehicle data (`vehicles.csv`), restricted to model years 2015 and later. Details, scope filter, limitations and licence notes: `data/README.md`. TODO (Aktore): add your own dataset discussion.

In [2]:
download_raw()          # no-op if already downloaded
raw = load_dataset()    # scope filter (2015+, no EV/PHEV/FCV) + schema check
print(raw.shape)
raw.head()

(13975, 14)


,row_id,model_year,make,model,mpg,cylinders,displacement,drive,vehicle_class,fuel_type,transmission,hybrid,turbo,supercharged
0,27058,2015,Nissan,GT-R,19,6.0,3.8,All-Wheel Drive,Subcompact Cars,Premium Gasoline,Automatic,0,1,0
1,27059,2015,Volvo,S60 AWD,23,5.0,2.5,All-Wheel Drive,Compact Cars,Regular Gasoline,Automatic,0,1,0
2,27060,2015,Mazda,6,30,4.0,2.5,Front-Wheel Drive,Midsize Cars,Regular Gasoline,Automatic,0,0,0
3,27061,2015,Mazda,6,32,4.0,2.5,Front-Wheel Drive,Midsize Cars,Regular Gasoline,Automatic,0,0,0
4,27062,2015,Mazda,6,29,4.0,2.5,Front-Wheel Drive,Midsize Cars,Regular Gasoline,Manual,0,0,0


## 3. EDA
**Status: TODO (Aktore)** - bring the four+ final plots and interpretations from `01_eda.ipynb` here (or embed the saved figures from `reports/figures/`). EDA must use the training split only.

## 4. Cleaning
Owner: Khamza. The scope filter and flag derivation are stateless and run before splitting (`data.prepare`). Remaining missing values would be *imputed inside the pipeline* (fitted on training rows only), so no further rows are dropped and nothing leaks.

In [3]:
raw_all = load_raw()
in_years = raw_all[raw_all["model_year"] >= 2015]
print(f"raw rows: {len(raw_all)} | model year >= 2015: {len(in_years)} | kept after scope filter: {len(raw)}")
print("removed as MPGe/CNG vehicles (atv_type):")
print(in_years[~in_years["row_id"].isin(raw["row_id"])]["atv_type"].value_counts())
print(f"rows that are exact duplicates of another row: {raw.duplicated(subset=[c for c in raw.columns if c != 'row_id']).sum()}")

raw rows: 50407 | model year >= 2015: 15994 | kept after scope filter: 13975
removed as MPGe/CNG vehicles (atv_type):
atv_type
EV                1544
Plug-in Hybrid     429
FCV                 41
Bifuel (CNG)         2
eFCV                 2
CNG                  1
Name: count, dtype: int64
rows that are exact duplicates of another row: 416


In [4]:
splits = make_splits(raw)
train, val, test = splits["train"], splits["val"], splits["test"]
print("Missing values in the training split (used columns):")
print(missing_summary(train).query("missing > 0") if missing_summary(train)["missing"].any() else "none")
train.describe().T[["min", "mean", "max"]]

Missing values in the training split (used columns):
none


,min,mean,max
row_id,27059.0,35460.434087,44909.0
model_year,2015.0,2020.301752,2027.0
mpg,9.0,23.207518,59.0
cylinders,3.0,5.566830,16.0
displacement,0.9,3.112826,8.4
hybrid,0.0,0.109617,1.0
turbo,0.0,0.549205,1.0
supercharged,0.0,0.046047,1.0


**Cleaning decisions and findings** (from the run on 2026-10-07, seed 42; rerun the cells to refresh):
- Of 50,407 raw rows, 15,994 are model year 2015 or later. 2,019 of those were removed because they are EV (1,544), plug-in hybrid (429), fuel-cell (43) or CNG (3) vehicles, whose efficiency is reported in MPGe, not MPG. **13,975 rows remain.**
- **No missing values** remain in the used columns, so the imputers in the pipeline are a safeguard (and are fitted on training data only if a gap ever appears in new data).
- `make`/`model` are not features (too many distinct values); `trany` is reduced to `transmission` (Automatic/Manual/Other).
- 416 rows are exact duplicates of another row; they are kept together in one split.
- **Plausibility check on the training split:** MPG 9-59, displacement 0.9-8.4 L, cylinders 3-16 (3, 4, 5, 6, 8, 10, 12, 16), all positive and consistent with real road vehicles. We found nothing to remove. Extreme values (16 cylinders, 59 MPG hybrids) are real cars, so they stay.

## 5. Feature engineering
Owner: Khamza. One stateless engineered feature, `displacement_per_cylinder` (litres per cylinder): engine size per cylinder should separate large-bore engines from small turbo engines better than either column alone. Categorical columns (`drive`, `transmission`, `vehicle_class`, `fuel_type`) are one-hot encoded.

We test the hypothesis with Linear Regression on **validation** data and with 5-fold CV on **training** data only (the test set is untouched).

In [5]:
add_features(split_xy(train)[0]).head()

,model_year,cylinders,displacement,hybrid,turbo,supercharged,drive,transmission,vehicle_class,fuel_type,displacement_per_cylinder
1,2015,5.0,2.5,0,1,0,All-Wheel Drive,Automatic,Compact Cars,Regular Gasoline,0.500000
2,2015,4.0,2.5,0,0,0,Front-Wheel Drive,Automatic,Midsize Cars,Regular Gasoline,0.625000
4,2015,4.0,2.5,0,0,0,Front-Wheel Drive,Manual,Midsize Cars,Regular Gasoline,0.625000
5,2015,6.0,3.8,0,0,0,Rear-Wheel Drive,Automatic,Large Cars,Regular Gasoline,0.633333
6,2015,8.0,5.0,0,0,0,Rear-Wheel Drive,Automatic,Large Cars,Premium Gasoline,0.625000


In [6]:
X_tr, y_tr = split_xy(train)
X_va, y_va = split_xy(val)
rows = {}
for label, kwargs in {"LR (base features)": {}, "LR + displacement_per_cylinder": {"engineered": True}}.items():
    p = build_linear_regression(**kwargs).fit(X_tr, y_tr)
    cv = cross_validate_pipeline(p, X_tr, y_tr)
    rows[label] = {**{f"val_{k}": v for k, v in evaluate(p, X_va, y_va).items()},
                   "cv_mae_mean": cv["mae_mean"], "cv_mae_std": cv["mae_std"]}
pd.DataFrame(rows).T.round(3)

,val_mae,val_rmse,val_r2,cv_mae_mean,cv_mae_std
LR (base features),2.132,2.873,0.785,2.086,0.015
LR + displacement_per_cylinder,2.019,2.778,0.799,1.987,0.007


**Outcome** (run of 2026-10-07, seed 42): adding `displacement_per_cylinder` lowered validation MAE from 2.13 to 2.02 MPG (RMSE 2.87 to 2.78, R² 0.785 to 0.799). Training-only 5-fold CV agrees: MAE 2.09 ± 0.015 vs 1.99 ± 0.007 MPG, and the improvement is far larger than the fold-to-fold spread. **Decision: keep the feature** (`engineered=True`). Caveat: this is one split of one dataset; the gain is about 0.1 MPG.

## 6. Splitting
70% train / 15% validation / 15% test, `random_state=42`, random split of records. Exact duplicate rows are grouped so they cannot cross splits. Note: near-identical variants of one car model can still land in different splits (see limitations in `data/README.md`). Train for fitting and EDA, validation for model selection, test only once at the end.

In [7]:
print({k: len(v) for k, v in splits.items()})
ids = {k: set(v["row_id"]) for k, v in splits.items()}
assert not (ids["train"] & ids["val"] or ids["train"] & ids["test"] or ids["val"] & ids["test"])
X_train, y_train = split_xy(train)
X_val, y_val = split_xy(val)
X_test, y_test = split_xy(test)  # not used until section 9

{'train': 9816, 'val': 2080, 'test': 2079}


## 7. Baseline - DummyRegressor (predicts the training mean MPG)
Any real model has to beat this.

In [8]:
baseline = build_baseline().fit(X_train, y_train)
results["Baseline (mean)"] = evaluate(baseline, X_val, y_val)
pd.DataFrame(results).T.rename(columns={"mae": "MAE (MPG)", "rmse": "RMSE (MPG)", "r2": "R2"})

,MAE (MPG),RMSE (MPG),R2
Baseline (mean),4.66176,6.196671,-1.294925e-07


**Interpretation:** the baseline always predicts the training mean MPG (about 23.2). On validation it is off by 4.66 MPG on average (MAE) with R² ≈ 0, as expected for a model that ignores every feature. Any real model must beat MAE 4.66 MPG; the spread of MPG in this data (std about 6.2) is the size of the problem.

## 8. Models

### Linear Regression
**Owner: Khamza** | Status: **DONE - awaiting teammate review**

Scaled numeric features + one-hot categoricals, with the engineered feature chosen in section 5. Ridge (alpha=1) was also tried as a check.

In [9]:
lr = build_linear_regression(engineered=True).fit(X_train, y_train)
results["Linear Regression"] = evaluate(lr, X_val, y_val)
ridge = build_linear_regression(engineered=True, alpha=1.0).fit(X_train, y_train)
print("Ridge(alpha=1) validation:", {k: round(v, 3) for k, v in evaluate(ridge, X_val, y_val).items()})
lr_cv = cross_validate_pipeline(lr, X_train, y_train)
print(f"LR 5-fold CV on train: MAE {lr_cv['mae_mean']:.3f} +/- {lr_cv['mae_std']:.3f} MPG")
print("train:", {k: round(v, 3) for k, v in evaluate(lr, X_train, y_train).items()})
pd.DataFrame(results).T.rename(columns={"mae": "MAE (MPG)", "rmse": "RMSE (MPG)", "r2": "R2"}).round(3)

Ridge(alpha=1) validation: {'mae': 2.021, 'rmse': 2.778, 'r2': 0.799}
LR 5-fold CV on train: MAE 1.987 +/- 0.007 MPG
train: {'mae': 1.977, 'rmse': 2.775, 'r2': 0.802}


,MAE (MPG),RMSE (MPG),R2
Baseline (mean),4.662,6.197,-0.000
Linear Regression,2.019,2.778,0.799


In [10]:
import matplotlib.pyplot as plt
names = lr.named_steps["prep"].get_feature_names_out()
coef = pd.Series(lr.named_steps["model"].coef_, index=names).sort_values()
show = pd.concat([coef.head(6), coef.tail(6)])
ax = show.plot.barh(figsize=(8, 5), title="Linear Regression: largest coefficients (MPG per 1 std / per category)")
ax.set_xlabel("effect on predicted MPG"); plt.tight_layout(); plt.show()
resid = y_val - lr.predict(X_val)
print("validation residual mean:", round(resid.mean(), 3), "| largest absolute error:", round(resid.abs().max(), 1), "MPG")

validation residual mean: -0.048 | largest absolute error: 18.2 MPG


/var/folders/0z/nyg58g9x17jfz79d4f4g3xjm0000gn/T/ipykernel_88329/3507914194.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.set_xlabel("effect on predicted MPG"); plt.tight_layout(); plt.show()


**Interpretation (run of 2026-10-07, seed 42):**
- Validation MAE **2.02 MPG**, RMSE 2.78 MPG, R² 0.80, versus baseline MAE 4.66 MPG: the model removes about 57% of the baseline's average error (4.66 → 2.02).
- Train MAE (1.98) and validation MAE (2.02) are close, and CV MAE is 1.99 ± 0.007, so there is no sign of overfitting; remaining error looks like **underfitting** (a linear model with no weight/horsepower), not variance.
- Ridge (alpha=1) gave the same result (2.02), so regularisation does not help here.
- **Sign sanity check:** larger displacement lowers predicted MPG, and vans/minivans/special-purpose vehicles have the most negative effects, which matches physical intuition. Diesel and compact/midsize cars raise it. Premium gasoline lowers it, probably because it marks performance cars, not because the fuel itself reduces MPG (the features are correlated, so individual coefficients should not be read as causal).
- Largest single validation error is 18.3 MPG, so some vehicles are predicted very badly; Askhat's error analysis should look at them.

### Decision Tree
**Owner: Askhat** | Status: **DONE - awaiting teammate review**

`build_decision_tree` = shared preprocessing (median imputation, one-hot; **no scaling**, because a tree only asks threshold questions such as "displacement <= 2.5 L?" on one feature at a time) + `DecisionTreeRegressor(random_state=42)`. Same feature set as Linear Regression (with `displacement_per_cylinder`). Each leaf predicts the mean MPG of the training cars that end up in it.

**Tuning (training data only):** grid over `max_depth` {4, 6, 8, 10, 12, None} x `min_samples_leaf` {1, 5, 10, 20} = 24 settings, 5-fold CV, MAE selects. Folds are `GroupKFold` on exact-duplicate groups, the same rule the shared split uses, so a duplicate car is never in both the fitting and scoring part of a fold. The pipeline (imputer, encoder, model) is refitted inside every fold.

In [11]:
from vehicle_efficiency.data import _duplicate_groups
from vehicle_efficiency.evaluation import (comparison_table, error_by_group, largest_errors,
                                           plot_actual_vs_predicted, plot_residuals, tune_pipeline)
from vehicle_efficiency.models import KNN_PARAM_GRID, TREE_PARAM_GRID

groups = _duplicate_groups(train)  # exact-duplicate groups inside the training split
cv_summary = {}  # model name -> CV result on identical folds

# Unrestricted tree (no depth or leaf limit) for reference: how far it fits the training data
full_tree = build_decision_tree(engineered=True).fit(X_train, y_train)
print("unrestricted tree  train MAE %.3f | validation MAE %.3f MPG | depth %d, leaves %d" % (
    evaluate(full_tree, X_train, y_train)["mae"], evaluate(full_tree, X_val, y_val)["mae"],
    full_tree.named_steps["model"].get_depth(), full_tree.named_steps["model"].get_n_leaves()))

tree, cv_summary["Decision Tree"], tree_grid = tune_pipeline(
    build_decision_tree(engineered=True), TREE_PARAM_GRID, X_train, y_train, groups=groups)
results["Decision Tree"] = evaluate(tree, X_val, y_val)
print("best:", cv_summary["Decision Tree"]["best_params"],
      "| CV MAE %.3f +/- %.3f" % (cv_summary["Decision Tree"]["cv_mae_mean"], cv_summary["Decision Tree"]["cv_mae_std"]),
      "| train MAE %.3f | validation MAE %.3f MPG" % (evaluate(tree, X_train, y_train)["mae"], results["Decision Tree"]["mae"]))
tree_grid.head(8).round(3)

unrestricted tree  train MAE 0.591 | validation MAE 1.059 MPG | depth 30, leaves 3393


best: {'max_depth': None, 'min_samples_leaf': 1} | CV MAE 1.090 +/- 0.024 | train MAE 0.591 | validation MAE 1.059 MPG


,params,cv_mae_mean,cv_mae_std,cv_rmse_mean,cv_r2_mean,train_mae_mean
0,"{'max_depth': None, 'min_samples_leaf': 1}",1.090,0.024,1.733,0.922,0.538
1,"{'max_depth': None, 'min_samples_leaf': 5}",1.157,0.022,1.655,0.929,0.914
2,"{'max_depth': 12, 'min_samples_leaf': 1}",1.204,0.030,1.713,0.924,0.956
3,"{'max_depth': 12, 'min_samples_leaf': 5}",1.254,0.025,1.721,0.924,1.096
4,"{'max_depth': None, 'min_samples_leaf': 10}",1.257,0.022,1.759,0.920,1.106
5,"{'max_depth': 12, 'min_samples_leaf': 10}",1.320,0.030,1.813,0.915,1.201
6,"{'max_depth': 10, 'min_samples_leaf': 1}",1.326,0.034,1.802,0.916,1.161
7,"{'max_depth': 10, 'min_samples_leaf': 5}",1.359,0.037,1.818,0.915,1.238


**Decision Tree interpretation** (run of 2026-10-09, seed 42):
- Cross-validation chose the **least restricted** tree in the grid: `max_depth=None`, `min_samples_leaf=1` (CV MAE 1.090 ± 0.024 MPG). Every added restriction made CV error worse (e.g. depth 12 → 1.204, depth 10 → 1.326).
- The tree grows to depth 30 with 3,393 leaves. Train MAE is 0.59 MPG versus validation MAE 1.06 MPG, so it clearly overfits: it fits the training cars almost exactly.
- It still has the lowest validation MAE of all models. A likely reason is that the data contains many near-identical variants of the same car (same model in different years or trims), and these can land in different splits (see limitations in `data/README.md`). Memorising a very similar training car is then a good strategy, so validation scores may be optimistic for truly new car designs.

### KNN
**Owner: Askhat** | Status: **DONE - awaiting teammate review**

`build_knn` = shared preprocessing **with `StandardScaler`** + `KNeighborsRegressor`. KNN predicts the (optionally distance-weighted) mean MPG of the *k* most similar training cars. "Similar" is a distance over all features, so without scaling the feature with the largest range (model year, ~2015-2027) would dominate and 0/1 columns would hardly count. The scaler sits **inside the Pipeline**, so its mean/std come from the fitting part of each fold only.

**Tuning:** `n_neighbors` {3, 5, 10, 15, 25} x `weights` {uniform, distance} x `p` {1 Manhattan, 2 Euclidean} = 20 settings, same grouped 5-fold CV on training data, MAE selects.

In [12]:
knn, cv_summary["KNN"], knn_grid = tune_pipeline(
    build_knn(engineered=True), KNN_PARAM_GRID, X_train, y_train, groups=groups)
results["KNN"] = evaluate(knn, X_val, y_val)
print("best:", cv_summary["KNN"]["best_params"],
      "| CV MAE %.3f +/- %.3f" % (cv_summary["KNN"]["cv_mae_mean"], cv_summary["KNN"]["cv_mae_std"]),
      "| train MAE %.3f | validation MAE %.3f MPG" % (evaluate(knn, X_train, y_train)["mae"], results["KNN"]["mae"]))

# Why scaling matters: best setting, but without the scaler
from vehicle_efficiency.models import make_pipeline
from sklearn.neighbors import KNeighborsRegressor
best_knn = {k: v for k, v in cv_summary["KNN"]["best_params"].items()}
unscaled = make_pipeline(KNeighborsRegressor(**best_knn), scale=False, engineered=True).fit(X_train, y_train)
print("same KNN without scaling: validation MAE %.3f MPG" % evaluate(unscaled, X_val, y_val)["mae"])
knn_grid.head(8).round(3)

/Users/askhat/projects/vehicle-fuel-efficiency-ml/.venv/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


best: {'n_neighbors': 3, 'p': 1, 'weights': 'uniform'} | CV MAE 1.112 +/- 0.033 | train MAE 0.776 | validation MAE 1.093 MPG
same KNN without scaling: validation MAE 1.376 MPG


,params,cv_mae_mean,cv_mae_std,cv_rmse_mean,cv_r2_mean,train_mae_mean
0,"{'n_neighbors': 3, 'p': 1, 'weights': 'uniform'}",1.112,0.033,1.650,0.930,0.781
1,"{'n_neighbors': 3, 'p': 1, 'weights': 'distance'}",1.117,0.033,1.710,0.924,0.559
2,"{'n_neighbors': 5, 'p': 1, 'weights': 'distance'}",1.121,0.030,1.686,0.926,0.543
3,"{'n_neighbors': 5, 'p': 1, 'weights': 'uniform'}",1.124,0.028,1.616,0.933,0.895
4,"{'n_neighbors': 3, 'p': 2, 'weights': 'uniform'}",1.128,0.026,1.665,0.928,0.786
5,"{'n_neighbors': 3, 'p': 2, 'weights': 'distance'}",1.138,0.031,1.727,0.923,0.559
6,"{'n_neighbors': 5, 'p': 2, 'weights': 'uniform'}",1.144,0.027,1.633,0.931,0.913
7,"{'n_neighbors': 5, 'p': 2, 'weights': 'distance'}",1.148,0.023,1.707,0.925,0.543


**KNN interpretation:**
- Best setting: `n_neighbors=3`, `weights=uniform`, `p=1` (Manhattan distance), CV MAE 1.112 ± 0.033 MPG, validation MAE 1.093 MPG. The smallest *k* in the grid won, which again shows that very local information (the few most similar cars) predicts MPG best on this data.
- Train MAE 0.78 vs validation 1.09: some overfitting, smaller than the tree's.
- **Scaling matters:** the same KNN without `StandardScaler` reaches validation MAE 1.376 MPG, about 26% worse than 1.093. Without scaling the distance is dominated by the features with large numeric ranges (model year, displacement), and 0/1 features such as `hybrid` barely count.

### Cross-validation
**Owner: Askhat** | Status: **DONE**

All four models on the **same** grouped 5-fold split of the training data (the tree and KNN numbers are their best grid settings). The spread (std) shows how much the score depends on which rows are in a fold. Note: section 8's Linear Regression CV (MAE 1.99) used shuffled `KFold`; the numbers here use grouped folds, so they can differ slightly.

In [13]:
for name, pipe in {"Baseline (mean)": build_baseline(),
                   "Linear Regression": build_linear_regression(engineered=True)}.items():
    _, cv_summary[name], _ = tune_pipeline(pipe, {}, X_train, y_train, groups=groups)
pd.DataFrame({n: {"CV MAE mean (MPG)": s["cv_mae_mean"], "CV MAE std": s["cv_mae_std"],
                  "CV RMSE mean (MPG)": s["cv_rmse_mean"], "CV R2 mean": s["cv_r2_mean"]}
              for n, s in cv_summary.items()}).T.sort_values("CV MAE mean (MPG)").round(3)

,CV MAE mean (MPG),CV MAE std,CV RMSE mean (MPG),CV R2 mean
Decision Tree,1.090,0.024,1.733,0.922
KNN,1.112,0.033,1.650,0.930
Linear Regression,1.986,0.036,2.787,0.800
Baseline (mean),4.635,0.067,6.233,-0.001


**Cross-validation interpretation:**
- On identical grouped folds, CV ranks the models the same way as the validation set: Decision Tree (1.090) ≈ KNN (1.112) < Linear Regression (1.986) < Baseline (4.635 MPG). CV and validation MAE differ by at most 0.03 MPG for the tree and KNN, so the validation numbers are not a lucky split.
- The standard deviations are small (0.02-0.07 MPG), so results are stable across folds.
- The tree-KNN difference (0.02 MPG in CV, 0.03 on validation) is about as large as the fold-to-fold spread, so the two models are **practically tied**. Both are clearly better than Linear Regression (about 0.9 MPG lower MAE).

## 9. Evaluation
**Owner: Askhat** | Status: **DONE - awaiting teammate review**

(1) Comparison on the validation split (all models fitted on train only). (2) The final model is the one with the lowest validation MAE. (3) Only then, one test evaluation. (4) Error analysis on validation. The same numbers are produced by `python -m vehicle_efficiency.compare` and saved in `reports/results/`; figures go to `reports/figures/`.

In [14]:
table = comparison_table(results)
best_name = table.index[0]
print("Selected by validation MAE:", best_name)
table.round(3)

Selected by validation MAE: Decision Tree


,MAE (MPG),RMSE (MPG),R2
Decision Tree,1.059,1.691,0.926
KNN,1.093,1.665,0.928
Linear Regression,2.019,2.778,0.799
Baseline (mean),4.662,6.197,-0.000


In [15]:
models = {"Linear Regression": lr, "Decision Tree": tree, "KNN": knn}
val_preds = {name: m.predict(X_val) for name, m in models.items()}
plot_actual_vs_predicted(y_val, val_preds, "actual_vs_predicted_validation.png"); plt.show()
plot_residuals(y_val, val_preds, "residuals_validation.png"); plt.show()

/var/folders/0z/nyg58g9x17jfz79d4f4g3xjm0000gn/T/ipykernel_88329/4294780319.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plot_actual_vs_predicted(y_val, val_preds, "actual_vs_predicted_validation.png"); plt.show()
/var/folders/0z/nyg58g9x17jfz79d4f4g3xjm0000gn/T/ipykernel_88329/4294780319.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plot_residuals(y_val, val_preds, "residuals_validation.png"); plt.show()


**Plots:**
- *Actual vs predicted:* Linear Regression's points bend away from the diagonal: it strongly under-predicts cars above about 40 MPG (it never predicts much more than 41). A straight-line model cannot follow the curved relation between engine size and MPG. Tree and KNN points sit much closer to the diagonal across the whole range.
- *Residuals:* Linear Regression shows a curved pattern (residuals are mostly positive at both very low and high predicted MPG), which means a systematic error from a non-linear relationship. Tree and KNN residuals form a band around 0, mostly within ±5 MPG, but both become much more spread out above about 40 MPG, where there are few cars, most of them hybrids.

In [16]:
best_pred = val_preds[best_name]
print(f"{best_name}: 10 largest validation errors (error = predicted - actual, MPG)")
largest_errors(val, y_val, best_pred)

Decision Tree: 10 largest validation errors (error = predicted - actual, MPG)


,make,model,model_year,vehicle_class,cylinders,displacement,drive,transmission,fuel_type,hybrid,turbo,actual_mpg,predicted_mpg,error
5266,Buick,LaCrosse eAssist,2019,Midsize Cars,4.0,2.5,Front-Wheel Drive,Automatic,Regular Gasoline,1,0,29,42.0,13.0
10809,Toyota,Prius,2023,Midsize Cars,4.0,2.0,Front-Wheel Drive,Automatic,Regular Gasoline,1,0,57,44.0,-13.0
13515,Mitsubishi,Outlander 2WD,2026,Small Sport Utility Vehicle 2WD,4.0,1.5,Front-Wheel Drive,Automatic,Regular Gasoline,1,1,28,39.5,11.5
4028,Toyota,Camry Hybrid LE,2018,Midsize Cars,4.0,2.5,Front-Wheel Drive,Automatic,Regular Gasoline,1,0,52,42.0,-10.0
5736,Toyota,Camry Hybrid LE,2019,Midsize Cars,4.0,2.5,Front-Wheel Drive,Automatic,Regular Gasoline,1,0,52,42.0,-10.0
6191,Toyota,Corolla Hybrid,2020,Compact Cars,4.0,1.8,Front-Wheel Drive,Automatic,Regular Gasoline,1,0,52,42.0,-10.0
10812,Toyota,Prius XLE/LTD,2023,Midsize Cars,4.0,2.0,Front-Wheel Drive,Automatic,Regular Gasoline,1,0,52,44.0,-8.0
6748,Ford,Fusion FWD,2020,Midsize Cars,4.0,2.5,Front-Wheel Drive,Automatic,Regular Gasoline,0,0,24,31.4,7.4
3612,Ford,C-MAX Hybrid FWD,2017,Large Cars,4.0,2.0,Front-Wheel Drive,Automatic,Regular Gasoline,1,0,40,47.0,7.0
4826,Toyota,Prius Eco,2018,Midsize Cars,4.0,1.8,Front-Wheel Drive,Automatic,Regular Gasoline,1,0,56,49.0,-7.0


In [17]:
error_by_group(val, y_val, best_pred, "vehicle_class").head(8)

,n,mae,mean_error
vehicle_class,,,
"Vans, Passenger Type",2,2.00,2.00
Small Sport Utility Vehicle 2WD,123,1.43,0.08
Compact Cars,228,1.43,0.02
Midsize Cars,277,1.39,0.07
Small Station Wagons,55,1.24,-0.06
Small Sport Utility Vehicle 4WD,255,1.20,-0.10
Midsize Station Wagons,18,1.06,0.28
Large Cars,137,1.04,0.10


**Error analysis (Decision Tree, validation):**
- 9 of the 10 largest errors are **hybrids**. The tree's MAE is 1.83 MPG on the 241 hybrids versus 0.96 MPG on non-hybrids.
- The cause is missing information: the only hybrid feature is a 0/1 flag. Training hybrids range from 11 to 59 MPG, so the flag cannot separate a mild hybrid (Buick LaCrosse eAssist, actual 29, predicted 42) from a full hybrid (Toyota Prius, actual 57, predicted 44). Errors go in both directions.
- By vehicle class, error is highest for compact, midsize and small SUV classes (about 1.4 MPG), where most hybrids are, and lowest for pickups and large SUVs (under 1 MPG).
- No class shows a large average bias (mean error within ±0.3 MPG for classes with more than 10 cars), so the model is not systematically too high or too low for a whole class.
- Improvement ideas: a feature for hybrid type or battery/electric motor size, and vehicle weight and horsepower, which this dataset does not contain.

### Final test evaluation (run once)
The model choice above is fixed before this cell. The test split has not been used for any decision.

In [18]:
RUN_FINAL_TEST = True  # team agreed on 2026-10-09: Decision Tree is the final model
if RUN_FINAL_TEST:
    from vehicle_efficiency.evaluation import save_results
    final_test = evaluate(models[best_name], X_test, y_test)
    save_results({"model": best_name, "selected_by": "lowest validation MAE",
                  "n_test": len(X_test), "units": "MAE and RMSE in miles per gallon; R2 unitless",
                  "test": final_test}, "final_test.json")
    print(best_name, "TEST:", {k: round(v, 3) for k, v in final_test.items()})
else:
    print("Test evaluation not run yet (RUN_FINAL_TEST = False).")

Decision Tree TEST: {'mae': 1.038, 'rmse': 1.668, 'r2': 0.928}


**Test result** (run once on 2026-10-09, after the team fixed the model choice):
- Decision Tree on the 2,079 test cars: **MAE 1.04 MPG**, RMSE 1.67 MPG, R² 0.93 (saved in `reports/results/final_test.json`).
- This is very close to its validation result (MAE 1.06, R² 0.93) and its CV estimate (1.09 ± 0.02), so the validation score was not a lucky split and the model generalises to unseen records from the same data.
- Compared with the baseline (validation MAE 4.66), the tree's average error is about 78% smaller.
- Caveat from the error analysis: unseen records still share car models with the training data, so error on completely new vehicle designs, and on hybrids, is likely higher.

## 10. Conclusions
**Status: NOT STARTED - all.** **Conclusions:** write only what the real results above support: best model, how much it improves on the baseline (in MPG), limitations (no weight/horsepower, only 2015+ vehicles, near-duplicate variants of the same model), what you would try next.

## 11. Final-stage plan
**Owner: Aktore** | Status: **NOT STARTED**. Summarise `docs/final_stage_plan.md` once written.